In [14]:
import intake, intake_esm


from IPython.display import display, HTML

from urllib.parse import quote

import os
import numpy as np

import pandas as pd
import json

In [15]:
## Fonctions pour faciliter la visualisation


def GetChildrenFiltre(df, filtres):  # fi : filtre name , l : is a leaf ?
    res = []
    if len(filtres) == 1:
        dfg = df.groupby(filtres[0])  # du fait que ce sont les derniere feuille, juste on les regoupe pour les compter
        for name, count in list(zip(list(dfg.nunique().index), list(dfg.nunique()["path"]))):
            res.append({"name": name, "size": count})

    else:  # on est en train d'applique un filtre au milieu
        for item in list(pd.unique(df[filtres[0]])):  # last = False
            dfFiltre = df.where(df[filtres[0]] == item).dropna(how="all")
            res.append({"name": item, "children": GetChildrenFiltre(dfFiltre, filtres[1:])})
    return res


def Trace(cat, filtres): ## Regroupe en bulles
    output = {"name": "root", "children": GetChildrenFiltre(df, filtres)}#cat.df, filtres)}
    # print(output)
    html_string = """
    <!DOCTYPE html>

    <meta content="utf-8" http-equiv="encoding">
    <style>

    .node {{
      cursor: pointer;
    }}

    .node:hover {{
      stroke: #000;
      stroke-width: 1.5px;
    }}

    .node--leaf {{
      fill: white;
    }}

    .label {{
      font: 20px "Helvetica Neue", Helvetica, Arial, sans-serif;
      text-anchor: middle;
      text-shadow: 0 1px 0 #fff, 1px 0 0 #fff, -1px 0 0 #fff, 0 -1px 0 #fff;
    }}

    .label,
    .node--root,
    .node--leaf {{
      pointer-events: none;
    }}

    </style>
    <script src="https://d3js.org/d3.v5.min.js"></script>


    <svg width="760" height="760"></svg>

    <script type="text/javascript">


    var svg = d3.select("svg"),
        margin = 20,
        diameter = +svg.attr("width"),
        g = svg.append("g").attr("transform", "translate(" + diameter / 2 + "," + diameter / 2 + ")");

    var color = d3.scaleSequential(d3.interpolateViridis)
        .domain([-4, 4]);

    var pack = d3.pack()
        .size([diameter - margin, diameter - margin])
        .padding(2);


    root = {mVar}
    root = d3.hierarchy(root)
      .sum(function(d) {{ return d.size; }})
      .sort(function(a, b) {{ return b.value - a.value; }});

    var focus = root,
      nodes = pack(root).descendants(),
      view;

    var circle = g.selectAll("circle")
    .data(nodes)
    .enter().append("circle")
      .attr("class", function(d) {{ return d.parent ? d.children ? "node" : "node node--leaf" : "node node--root"; }})
      .style("fill", function(d) {{ return d.children ? color(d.depth) : null; }})
      .on("click", function(d) {{ if (focus !== d) zoom(d), d3.event.stopPropagation(); }});

    var text = g.selectAll("text")
    .data(nodes)
    .enter().append("text")
      .attr("class", "label")
      .style("fill-opacity", function(d) {{ return d.parent === root ? 1 : 0; }})
      .style("display", function(d) {{ return d.parent === root ? "inline" : "none"; }})
      .text(function(d) {{ return d.data.name; }});

    var node = g.selectAll("circle,text");

    svg
      .style("background", color(-1))
      .on("click", function() {{ zoom(root); }});

    zoomTo([root.x, root.y, root.r * 2 + margin]);

    function zoom(d) {{
    var focus0 = focus; focus = d;

    var transition = d3.transition()
        .duration(d3.event.altKey ? 7500 : 750)
        .tween("zoom", function(d) {{
          var i = d3.interpolateZoom(view, [focus.x, focus.y, focus.r * 2 + margin]);
          return function(t)  {{ zoomTo(i(t)); }};
        }});

    transition.selectAll("text")
      .filter(function(d) {{ return d.parent === focus || this.style.display === "inline"; }})
        .style("fill-opacity", function(d) {{ return d.parent === focus ? 1 : 0; }})
        .on("start", function(d) {{ if (d.parent === focus) this.style.display = "inline"; }})
        .on("end", function(d) {{ if (d.parent !== focus) this.style.display = "none"; }});
    }}

    function zoomTo(v) {{
    var k = diameter / v[2]; view = v;
    node.attr("transform", function(d) {{ return "translate(" + (d.x - v[0]) * k + "," + (d.y - v[1]) * k + ")"; }});
    circle.attr("r", function(d) {{ return d.r * k; }});
    }}


    </script>
    """

    # print(html_string)
    html_string = html_string.format(mVar=str(output))

    html_string = quote(html_string, safe="")

    # <script>document.body.onload = function () {{ {code} }}</script>
    # .format(code = js_string,safe="")
    # print(html_string)
    iframe = """<iframe 
        width="800"
        height="800"
        src="data:text/html;charset=UTF-8,{srcHmtl}">
    </iframe>
    """.format(srcHmtl=html_string)

    display(HTML(iframe))

In [16]:
start_list = ['1971-01-01T00:00:00', '1976-01-01T00:00:00', '1981-01-01T00:00:00', '1986-01-01T00:00:00', '1991-01-01T00:00:00', '1996-01-01T00:00:00', '2001-01-01T00:00:00', '2006-01-01T00:00:00', '2011-01-01T00:00:00', '2016-01-01T00:00:00', '2021-01-01T00:00:00', '2026-01-01T00:00:00', '2031-01-01T00:00:00', '2036-01-01T00:00:00', '2041-01-01T00:00:00', '2046-01-01T00:00:00', '2051-01-01T00:00:00', '2056-01-01T00:00:00', '2061-01-01T00:00:00', '2066-01-01T00:00:00', '2071-01-01T00:00:00', '2076-01-01T00:00:00', '2081-01-01T00:00:00', '2086-01-01T00:00:00', '2091-01-01T00:00:00','2096-01-01T00:00:00']

In [17]:
#### Ouvre le fichier .json, et permet d'y effectuer des recherches, selon la variable, le domaine, la fréquence, etc.

choice = "CORDEX_ADJUST" #'CORDEX' or 'CORDEX_ADJUST'

if choice == 'CORDEX':
    CordexCat = intake.open_esm_datastore("CORDEX.json").search(variable = ['tasmax']).search(domain= 'EUR-11').search(time_frequency = 'day').search(experiment = ['rcp45','rcp85', 'historical']).search(latest = 'True').search(ensemble = ['r1i1p1','r12i1p1']).search(period_start = start_list)#.search(ensemble = 'r1i1p1').search(rcm_version = 'v1')#.search(period_start = ['1950-01-01T00:00:00'])#.search(experiment = 'evaluation')##
elif choice == 'CORDEX_ADJUST':
    CordexCat = intake.open_esm_datastore("CORDEX_ADJUST.json").search(variable = ['tasmax']).search(experiment = ['rcp45','rcp85', 'historical'])




In [18]:
CordexCat.df

,,path,project,product,domain,driving_model,experiment,ensemble,rcm_model,rcm_version,time_frequency,variable,version,period_start,period_end,latest,correction,reference_dataset
0,4,/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CN...,CORDEX,ADJUST,EUR-11,CNRM-CERFACS-CNRM-CM5,historical,r1i1p1,CNRM-ALADIN63,BC2602,day,tasmax,BC2602,1951-01-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
1,10,/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CN...,CORDEX,ADJUST,EUR-11,CNRM-CERFACS-CNRM-CM5,rcp45,r1i1p1,CNRM-ALADIN63,BC2602,day,tasmax,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
2,16,/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CN...,CORDEX,ADJUST,EUR-11,CNRM-CERFACS-CNRM-CM5,rcp85,r1i1p1,CNRM-ALADIN63,BC2602,day,tasmax,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
3,22,/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/...,CORDEX,ADJUST,EUR-11,ICHEC-EC-EARTH,historical,r1i1p1,KNMI-RACMO22E,BC2602,day,tasmax,BC2602,1950-01-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
4,28,/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/...,CORDEX,ADJUST,EUR-11,ICHEC-EC-EARTH,rcp45,r1i1p1,KNMI-RACMO22E,BC2602,day,tasmax,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
5,34,/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/...,CORDEX,ADJUST,EUR-11,ICHEC-EC-EARTH,rcp85,r1i1p1,KNMI-RACMO22E,BC2602,day,tasmax,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
6,40,/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/...,CORDEX,ADJUST,EUR-11,ICHEC-EC-EARTH,historical,r12i1p1,MOHC-HadREM3-GA7-05,BC2602,day,tasmax,BC2602,1951-12-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
7,46,/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/...,CORDEX,ADJUST,EUR-11,ICHEC-EC-EARTH,rcp85,r12i1p1,MOHC-HadREM3-GA7-05,BC2602,day,tasmax,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
8,52,/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/...,CORDEX,ADJUST,EUR-11,ICHEC-EC-EARTH,historical,r1i1p1,SMHI-RCA4,BC2602,day,tasmax,BC2602,1970-01-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
9,58,/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/...,CORDEX,ADJUST,EUR-11,ICHEC-EC-EARTH,rcp85,r1i1p1,SMHI-RCA4,BC2602,day,tasmax,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5


In [19]:
CordexCat.df['directory']=None
array = np.array([(0,0)]*len(CordexCat.df))
for i in CordexCat.df.index :
    CordexCat.df.loc[i,'directory'] = os.path.dirname(CordexCat.df.loc[i,'path'])

df = CordexCat.df
df.to_csv(f"/home/user/These/cordex_htws_cc3d/export_{choice}.csv")

In [20]:
# Afficher l'arborescence en "bulles" imbriquées :

Trace(df,['driving_model', 'rcm_model', 'variable','ensemble','experiment','period_start'])

In [21]:
sub_df = df[['driving_model','rcm_model','experiment','directory','rcm_version']].drop_duplicates()
pairs_dict = {}
for i in sub_df.index :
    if sub_df.loc[i,"experiment"]=="rcp85" or sub_df.loc[i,"experiment"]=="rcp45" :
        try :
            historical_dir = sub_df[(sub_df["driving_model"]==sub_df.loc[i,"driving_model"])&(sub_df["rcm_model"]==sub_df.loc[i,"rcm_model"])&(sub_df["rcm_version"]==sub_df.loc[i,"rcm_version"])&(sub_df["experiment"]=="historical")]["directory"].values[0]
        except :
            pass
        pairs_dict[(sub_df.loc[i,"driving_model"],sub_df.loc[i,"rcm_model"],sub_df.loc[i,"experiment"],sub_df.loc[i,"rcm_version"])]={"rcp":sub_df.loc[i,"directory"],"historical":historical_dir}

In [22]:
def json_dumps_dict(dict_having_tuple_as_key):
    if not isinstance(dict_having_tuple_as_key, dict):
        raise Exception('Error using json_dumps_dict_having_tuple_as_key: The input variable is not a dictionary.')  
    list_of_dicts_having_key_and_value_as_keys = [{'key': k, 'value': v} for k, v in dict_having_tuple_as_key.items()]
    json_array_having_key_and_value_as_keys = json.dumps(list_of_dicts_having_key_and_value_as_keys)
    return json_array_having_key_and_value_as_keys

def json_loads_dictionary(json_array_having_key_and_value_as_keys):
    list_of_dicts_having_key_and_value_as_keys = json.loads(json_array_having_key_and_value_as_keys)
    if not all(['key' in diz for diz in list_of_dicts_having_key_and_value_as_keys]) and all(['value' in diz for diz in list_of_dicts_having_key_and_value_as_keys]):
        raise Exception('Error using json_loads_dictionary_split_into_key_and_value_as_keys_and_underwent_json_dumps: at least one dictionary in list_of_dicts_having_key_and_value_as_keys ismissing key "key" or key "value".')
    dict_having_tuple_as_key = {}
    for dict_having_key_and_value_as_keys in list_of_dicts_having_key_and_value_as_keys:
        dict_having_tuple_as_key[ tuple(dict_having_key_and_value_as_keys['key']) ] = dict_having_key_and_value_as_keys['value']
    return dict_having_tuple_as_key

In [23]:
dumped_dict = json_dumps_dict(pairs_dict)
with open(f'{choice}_pairs_path_dict.json', 'w') as fp:
    json.dump(dumped_dict, fp)

with open(f'{choice}_pairs_path_dict.json', 'r') as f:
    data = json.load(f)

loaded_dict = json_loads_dictionary(data)

In [24]:
loaded_dict

{('CNRM-CERFACS-CNRM-CM5',
  'CNRM-ALADIN63',
  'rcp45',
  'BC2602'): {'rcp': '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63/rcp45/r1i1p1/day/tasmaxAdjust', 'historical': '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63/historical/r1i1p1/day/tasmaxAdjust'},
 ('CNRM-CERFACS-CNRM-CM5',
  'CNRM-ALADIN63',
  'rcp85',
  'BC2602'): {'rcp': '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63/rcp85/r1i1p1/day/tasmaxAdjust', 'historical': '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63/historical/r1i1p1/day/tasmaxAdjust'},
 ('ICHEC-EC-EARTH',
  'KNMI-RACMO22E',
  'rcp45',
  'BC2602'): {'rcp': '/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/KNMI-RACMO22E/rcp45/r1i1p1/day/tasmaxAdjust', 'historical': '/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/KNMI-RACMO22E/historical/r1i1p1/day/tasmaxAdjust'},
 ('ICHEC-EC-EARTH',
  'KNMI-RACMO22E',
  'rcp85',
  'BC2602'): {'rcp': '/scratchu/yrobin/BC2602/ADJUST/ICHEC-EC-EARTH/KNMI-RACM

In [25]:
import os

def json_loads_dictionary(json_array_having_key_and_value_as_keys):
    list_of_dicts_having_key_and_value_as_keys = json.loads(json_array_having_key_and_value_as_keys)
    if not all(['key' in diz for diz in list_of_dicts_having_key_and_value_as_keys]) and all(['value' in diz for diz in list_of_dicts_having_key_and_value_as_keys]):
        raise Exception('Error using json_loads_dictionary_split_into_key_and_value_as_keys_and_underwent_json_dumps: at least one dictionary in list_of_dicts_having_key_and_value_as_keys ismissing key "key" or key "value".')
    dict_having_tuple_as_key = {}
    for dict_having_key_and_value_as_keys in list_of_dicts_having_key_and_value_as_keys:
        dict_having_tuple_as_key[ tuple(dict_having_key_and_value_as_keys['key']) ] = dict_having_key_and_value_as_keys['value']
    return dict_having_tuple_as_key

with open(f'{choice}_pairs_path_dict.json', 'r') as f:
    data = json.load(f)

loaded_dict = json_loads_dictionary(data)

for k,v in loaded_dict.items() :
    os.system(f"sbatch test_job.sh {k[0]} {k[1]} {k[2]} {k[3]} {v['historical']} {v['rcp']}")
    

sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
sh: 1: sbatch: not found
